# Maternal Engagement EF — On-nest vs Off-nest (1-Hz frequency)

Paper-active dCSFA-NMF model for on-nest vs off-nest LFP windows in 1 Hz frequency resolution.

* Final model: `Maternal_model_1Hz_onnest_ver3.pt`
* Hyperparameters from LOO validation (sup_weight=0.045, n_epochs=300, batch_size=256, h=128, seed=2025)
* Stage backproject artifact: `OnnestEF_1Hz.xlsx`

In [ ]:
# Allow imports from ../src
import sys, os, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if os.path.join(_repo_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_repo_root, "src"))

from electome.data_utils import create_period_dataset, create_split_dataset, clean_mouse_id
from electome.training import run_loo_cv, train_final_model
from electome.analysis import process_W_nmf_dual_filter
from electome.viz import create_bar_heatmap_selective
from electome.workflow import validate_on_ELS, run_circos_prep, run_stage_backproject
from electome.additional_analyses import pup_retrieval_export

from electome.dCSFA_NMF_Ver1 import dCSFA_NMF as DCSFA_VER1


In [ ]:
# Add 1Hz-aware extra imports
from electome.sara_requests import onnest_loading_export, p3_behavior_export

from electome.dCSFA_NMF_Ver1 import dCSFA_NMF as DCSFA_VER1


## Section 1. Data loading and processing

In [ ]:
TRAINING_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_onnest_spec_features_Trim.pkl"

# Raw 1Hz mouse-ids include the Mouse prefix and F# face/cohort marker.
C_MOUSE_IDS_RAW = ["MouseC7ELS11", "MouseC2F4ELS18", "MouseC5F4ELS20", "MouseC7F1ELS22",
                    "MouseC1F3ELS32", "MouseC5F3ELS40", "MouseC6F4ELS42", "MouseC7F4ELS45"]
E_MOUSE_IDS_RAW = ["MouseE1F4ELS33", "MouseE2ELS3", "MouseE3F1ELS37", "MouseE4F2ELS39",
                    "MouseE5F1ELS41", "MouseE6F4ELS44"]

# Canonical ids for the stage-backproject section
C_MOUSE_IDS = [clean_mouse_id(m) for m in C_MOUSE_IDS_RAW]
E_MOUSE_IDS = [clean_mouse_id(m) for m in E_MOUSE_IDS_RAW]

PERIODS_TO_KEEP = ["P1", "P3", "P8"]

with open(TRAINING_DATA_FILE, "rb") as f:
    train_dict = pickle.load(f)

base_mask = np.isin(train_dict["period"], PERIODS_TO_KEEP)
base_data = {k: v[base_mask] for k, v in train_dict.items()
             if isinstance(v, np.ndarray) and len(v) == len(train_dict["period"])}
base_y = base_data["onnest_label"]

train_c     = create_period_dataset(base_data, base_y, C_MOUSE_IDS_RAW, ["P1", "P3"], "Train C",     verbose=False)
test_c_P8   = create_period_dataset(base_data, base_y, C_MOUSE_IDS_RAW, ["P8"],       "Test C P8",   verbose=False)
test_e_P1P3 = create_period_dataset(base_data, base_y, E_MOUSE_IDS_RAW, ["P1", "P3"], "Test E P1P3", verbose=False)
test_e_P8   = create_period_dataset(base_data, base_y, E_MOUSE_IDS_RAW, ["P8"],       "Test E P8",   verbose=False)

print(f"Train C     : X={train_c['X'].shape},     mice={list(train_c['mouse_list'])}")
print(f"Test C P8   : X={test_c_P8['X'].shape},   mice={list(test_c_P8['mouse_list'])}")
print(f"Test E P1P3 : X={test_e_P1P3['X'].shape}, mice={list(test_e_P1P3['mouse_list'])}")
print(f"Test E P8   : X={test_e_P8['X'].shape},   mice={list(test_e_P8['mouse_list'])}")


## Section 2. LOO training

In [ ]:
SEED = 2025
N_EPOCHS = 300
N_PRE_EPOCHS = 100
NMF_MAX_ITER = 500
BATCH_SIZE = 256
LR = 1e-3

MODEL_PARAMS = {
    "n_components": 10,
    "n_sup_networks": 1,
    "optim_name": "SGD",
    "recon_loss": "MSE",
    "sup_recon_weight": 0.0,
    "sup_weight": 0.045,
    "phi_weight": 0,
    "n_intercepts": 1,
    "useDeepEnc": True,
    "h": 128,
    "sup_recon_type": "Residual",
    "feature_groups": None,
    "group_weights": None,
    "fixed_corr": "Positive",
    "momentum": 0.9,
    "sup_smoothness_weight": 1,
}

loo = run_loo_cv(
    train_c["X"], train_c["y"], train_c["y_intercept"],
    model_params=MODEL_PARAMS,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    n_pre_epochs=N_PRE_EPOCHS, nmf_max_iter=NMF_MAX_ITER,
    seed=SEED, n_jobs=4,
    dCSFA_NMF_class=DCSFA_VER1,
)
print(loo.summary())
print()
print(loo.per_mouse_table())


## Section 3. Full training (paper model)

In [ ]:
MODEL_SAVE_FILE = "Maternal_model_1Hz_onnest_ver3.pt"
MODEL_STATE_DICT = "Maternal_sd_1Hz_onnest_ver3.pt"

model = train_final_model(
    train_c["X"], train_c["y"], train_c["y_sampling"],
    model_params=MODEL_PARAMS,
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    n_pre_epochs=N_PRE_EPOCHS, nmf_max_iter=NMF_MAX_ITER,
    seed=SEED,
    save_to=MODEL_SAVE_FILE,
    state_dict_to=MODEL_STATE_DICT,
    dCSFA_NMF_class=DCSFA_VER1,
)
train_aucs = [auc[0] for auc in model.train_auc_hist]
print(f"Paper model saved : {MODEL_SAVE_FILE}")
print(f"  Final train AUC : {train_aucs[-1]:.4f}")


## Section 4. Circos plot

In [ ]:
df_circos = run_circos_prep(
    model, train_dict,
    output_csv="OnnestEF_1Hz_circos_input.csv",
    k=0, threshold_ratio=0.8,
)


## Section 5. Elements selection

In [ ]:
abs_cut, rel_cut, both_cut, abs_full, rel_full = process_W_nmf_dual_filter(
    model.get_W_nmf(), train_dict,
    abs_cum_ratio=0.9, rel_val=0.5,
    verbose=False,
)
fig = create_bar_heatmap_selective(abs_full, abs_cut, rel_full, rel_cut, both_cut)
fig.savefig("OnnestEF_1Hz_bar_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Bar heatmap saved : OnnestEF_1Hz_bar_heatmap.png  "
      f"({(~both_cut.isna()).sum().sum()} optimal features)")


## Section 6. Validation on ELS group

In [ ]:
els = validate_on_ELS(model, {
    "C mice P8 (held-out)": test_c_P8,
    "E mice P1+P3":         test_e_P1P3,
    "E mice P8":            test_e_P8,
})
print(els.summary())


## Section 8. Additional backprojections (additional backprojection)

In [ ]:
PUP_RETRIEVAL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P4_pup_retrieval_detail.pkl"

pup_retrieval_export(
    model,
    pup_retrieval_data_file=PUP_RETRIEVAL_DATA_FILE,
    c_mouse_ids=["C6ELS9"] + C_MOUSE_IDS,
    e_mouse_ids=E_MOUSE_IDS,
    output_xlsx="Onnest_pups_1Hz.xlsx",
)


In [ ]:
onnest_loading_export(
    model,
    training_data_file=TRAINING_DATA_FILE,
    output_xlsx="OnNestEF_OnOff_1Hz.xlsx",
)


In [ ]:
P3_BEHAVIOR_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_complete_trimmed.pkl"

p3_behavior_export(
    model,
    training_data_file=P3_BEHAVIOR_DATA_FILE,
    output_xlsx="OnNestEF_Behavior_P3_1Hz.xlsx",
)
